# Exercise 1: Embeddings and distances

Clustering aerial scenes with embeddings from a pre-trained vision transformer.

Two inputs, both from HuggingFace:

* **the images** - `blanchon/UC_Merced`, 2,100 aerial scenes in 21 classes
* **the embeddings** - `MustafaArikan/vit_embeddings_uc_merced_layers`, embedding vectors taken
  from 7 depths of a ViT (768 dims each)

## 1. Load the dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from huggingface_hub import hf_hub_download

DATA = "blanchon/UC_Merced"

ds = load_dataset(DATA, split="train")
class_names = ds.features["label"].names
labels = np.array(ds["label"])

print(f"{len(ds)} images, {len(class_names)} classes")
print(class_names)

## 2. Embeddings

In [ ]:
order = np.asarray(sorted(range(len(ds)), key=lambda i: (class_names[labels[i]], i)))
y_all = labels[order] + 1                      # 1-based, as in the embeddings

rng = np.random.default_rng(0)
train_idx, test_idx = [], []
for label in range(1, len(class_names) + 1):
    idx = np.where(y_all == label)[0]
    rng.shuffle(idx)
    n_test = max(1, int(round(len(idx) * 0.2)))
    test_idx += list(idx[:n_test])
    train_idx += list(idx[n_test:])

train_idx, test_idx = np.asarray(train_idx), np.asarray(test_idx)

print(f"train {len(train_idx)} / test {len(test_idx)}")

## 3. Align the training images

Some of the 2,100 images are a few pixels off 256 x 256, so they are resized here;
`np.stack` would fail otherwise.

In [ ]:
def to_rgb(row):
    im = row["image"].convert("RGB")
    if im.size != (256, 256):
        im = im.resize((256, 256))
    return np.asarray(im)

X_train_rgb = np.stack([to_rgb(ds[int(i)]) for i in order[train_idx]])
y_train = y_all[train_idx]

print(X_train_rgb.shape, X_train_rgb.dtype)
print(y_train.shape)

## 4. Load the embeddings

Seven depths are available. Index `0` is the patch embedding, before any attention;
`12` is the output of the last block.

In [ ]:
REPO = "MustafaArikan/vit_embeddings_uc_merced_layers"
path = hf_hub_download(REPO, "vit_embeddings_uc_merced_layers.npz", repo_type="dataset")
d = np.load(path)

LAYERS = [0, 1, 2, 3, 6, 9, 12]
E_train = {k: d[f"E_train_l{k}"] for k in LAYERS}
E_test = {k: d[f"E_test_l{k}"] for k in LAYERS}

print("layers:", LAYERS)
print("dims  :", {k: v.shape for k, v in E_train.items()})
print("rows match the images:", np.array_equal(y_train, d["y_train"]))

## 5. The 10 farthest neighbours of one image

Cosine similarity against every other training image, sorted ascending, take the first 10.
"Farthest" means the images the model considers least alike.

In [ ]:
LAYER = 3                 # 0 = patch embeddings, 12 = last block
q = 144                   # query image index

E = E_train[LAYER]

norms = np.linalg.norm(E, axis=1)
sims = (E @ E[q]) / (norms * norms[q])

others = np.arange(len(E)) != q
farthest = np.where(others)[0][np.argsort(sims[others])][:10]

query_class = class_names[y_train[q] - 1]
print(f"layer {LAYER} | query: image {q}  ({query_class})\n")

for rank, j in enumerate(farthest, 1):
    print(f"  {rank:2}. image {j:5}  {class_names[y_train[j] - 1]:16} {sims[j]:+.3f}")

fig, axes = plt.subplots(1, 11, figsize=(20, 2.6),
                         gridspec_kw={"width_ratios": [1.4] + [1] * 10})

axes[0].imshow(X_train_rgb[q])
axes[0].set_title(f"query\n{query_class}", fontsize=9)
axes[0].axis("off")

for ax, j in zip(axes[1:], farthest):
    ax.imshow(X_train_rgb[j])
    ax.set_title(f"{class_names[y_train[j] - 1]}\n{sims[j]:+.2f}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. TODO: the 10 nearest neighbours

Same computation, sorted the other way; take the first 10.

Then compare with the grid above:

* do the nearest neighbours share the query's class?
* how much more alike are they than the farthest ones?

In [ ]:
# TODO: top 10 nearest

## 7. TODO: the same, for another layer

Repeat the 10 nearest for a different depth: try `LAYER = 1`, `6` and `12`.

Which depth returns neighbours that actually share the class?

In [ ]:
# TODO: top 10 nearest, another layer

## 8. TODO: another similarity measure

Cosine compares direction only and ignores magnitude. What else could you use to compare
two 768-dimensional vectors?

Find one alternative and show its 10 nearest for the same image:

e.g.
* Manhattan (L1), or correlation distance, does the ranking change?

Say in a sentence what each one assumes about the vectors.

In [ ]:
# TODO: alternative similarity measure